## Data creation — merged.csv

Builds the main dataset: daily weather (MET) joined to county-level detrended corn yield.

**Inputs** (under `data/raw/` by default):
- Yield: `corn/{IA,IL,...}.csv` (NASS-style: commodity_desc, statisticcat_desc, util_practice_desc, location_desc, year, Value)
- MET: `MET/met_{state}.csv` (id, date, tmax, tmin, prcp, snow, ...)
- `ghcnd-stations.txt` (fixed-width station list with id, lat, lon)
- County shapefile: `cb_2021_us_county_500k/cb_2021_us_county_500k.shp`

**Output:** `data/merged.csv` (and optionally cluster CSVs in a later section).

### 1. Config and paths

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

from dawn_config import (
    DATA_DIR,
    RAW_DATA_DIR,
    RAW_YIELD_DIR,
    RAW_MET_DIR,
    STATIONS_FILE,
    COUNTY_SHP_PATH,
    CORN_BELT_STATES,
)

# Ensure output dir exists
DATA_DIR.mkdir(parents=True, exist_ok=True)

### 2. Load shapefile and define helpers

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

all_counties = gpd.read_file(COUNTY_SHP_PATH).to_crs("EPSG:4326")

def load_yield_data(state_abbr, yield_dir):
    path = yield_dir / f"{state_abbr}.csv"
    df = pd.read_csv(path)
    df = df[
        (df["commodity_desc"] == "CORN") &
        (df["statisticcat_desc"] == "YIELD") &
        (df["util_practice_desc"] == "GRAIN")
    ].copy()
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
    df = df.dropna(subset=["Value"])
    return df

def detrend_yield(df, state_abbr):
    out = []
    for location, group in df.groupby("location_desc"):
        group = group.sort_values("year")
        x, y = group["year"], group["Value"]
        group = group.assign(
            detrended_yield=y - np.polyval(np.polyfit(x, y, 1), x),
            county_name=group["location_desc"].str.split(",").str[-1].str.strip().str.title(),
            state=state_abbr,
        )
        out.append(group)
    return pd.concat(out)

def load_met_data(state_abbr, met_dir):
    return pd.read_csv(met_dir / f"met_{state_abbr}.csv")

def assign_stations_to_counties(met_df, all_counties_gdf, state_fips, state_abbr, stations_path):
    cols = ["id", "latitude", "longitude", "elevation", "state", "name"]
    colspecs = [(0, 11), (12, 20), (21, 30), (31, 37), (38, 40), (41, 71)]
    stations = pd.read_fwf(stations_path, colspecs=colspecs, names=cols)
    stations = stations[stations["id"].isin(met_df["id"].unique())].copy()
    stations["geometry"] = stations.apply(
        lambda r: Point(r["longitude"], r["latitude"]), axis=1
    )
    stations_gdf = gpd.GeoDataFrame(stations, geometry="geometry", crs="EPSG:4326")
    state_counties = all_counties_gdf[all_counties_gdf["STATEFP"] == state_fips]
    joined = gpd.sjoin(stations_gdf, state_counties, how="left", predicate="within")
    county_lookup = joined[["id", "name", "latitude", "longitude", "NAME"]].rename(
        columns={"NAME": "county_name"}
    )
    merged = pd.merge(met_df, county_lookup, on="id", how="left")
    return merged.assign(state=state_abbr)


### 3. Build merged and save

In [ ]:
all_detrended = []
all_met_with_counties = []

print("Processing:", " ".join(CORN_BELT_STATES.keys()))
for state_abbr, fips in CORN_BELT_STATES.items():
    yield_df = load_yield_data(state_abbr, RAW_YIELD_DIR)
    yield_detrended = detrend_yield(yield_df, state_abbr)
    all_detrended.append(yield_detrended)
    met_df = load_met_data(state_abbr, RAW_MET_DIR)
    met_with_counties = assign_stations_to_counties(
        met_df, all_counties, fips, state_abbr, STATIONS_FILE
    )
    all_met_with_counties.append(met_with_counties)

detrended_df = pd.concat(all_detrended, ignore_index=True)
met_df = pd.concat(all_met_with_counties, ignore_index=True)
met_df["date"] = pd.to_datetime(met_df["date"])
met_df["year"] = met_df["date"].dt.year
met_df["month"] = met_df["date"].dt.month
met_df["day"] = met_df["date"].dt.day

merged = pd.merge(
    met_df,
    detrended_df[["county_name", "year", "detrended_yield", "state"]],
    on=["county_name", "year", "state"],
    how="left",
).dropna(subset=["detrended_yield"])

merged.to_csv(DATA_DIR / "merged.csv", index=False)
years = sorted(merged["year"].unique())
print(f"merged: shape={merged.shape}, years={years[0]}..{years[-1]}, #counties={merged['county_name'].nunique()}")
print(f"Saved to {DATA_DIR / 'merged.csv'}")

### 4. Optional: save cluster CSVs

Assign fixed state groups (same as in `dawn_mapping` and `dawn_models_clean`) and save one CSV per cluster to `data/` for modeling.

In [ ]:
# Fixed state groups used in dawn_models_clean (same as CLUSTER_DEFS in dawn_mapping)
CLUSTER_DEFS = {0: ["MI", "MN", "WI"], 1: ["IN", "KS", "MO", "NE", "OH"], 2: ["IA", "IL"]}

def state_to_cluster(s):
    for c, states in CLUSTER_DEFS.items():
        if s in states:
            return c
    return None

merged["cluster"] = merged["state"].map(state_to_cluster)
merged_clustered = merged.dropna(subset=["cluster"]).astype({"cluster": int})

for c, states in CLUSTER_DEFS.items():
    subset = merged_clustered[merged_clustered["cluster"] == c]
    name = f"cluster{c}_{'_'.join(states)}.csv"
    subset.drop(columns=["cluster"]).to_csv(DATA_DIR / name, index=False)
    print(f"  {name}: shape={subset.shape}")